<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">01 Data and Signals</h1>
<hr>


## **6. Redundancy**

<div class="alert alert-light">

  **Name:** Stefan Sykes
  
  **Matriculation Number:** 11-250-693

</div>


### 6.1 From Entropy to Redundancy

The previous notebook, [Entropy](01.5_Entropy.ipynb), distinguished three quantities: the information content $I(x)$ of one outcome, the symbol entropy $H(X)$ as its probability-weighted average, and the average encoded length $L$. We now ask how many bits we could avoid storing or transmitting while preserving the original sample values.

**Redundancy is structure that a lossless representation can exploit to use fewer bits.** Two examples from the entropy notebook show where it can arise:

- **Unequal symbol probabilities:** the eight three-bit symbols `000` through `111` need $L=3$ bit per source symbol in a fixed-length representation. In the unequal-probability example, $H(X)\approx2.04$ bit per source symbol. Assigning shorter codewords to frequent symbols reduces the average length to $L=2.05$.
- **Predictable symbol order:** an ordered sequence and a shuffled sequence can have the same symbol counts and therefore the same estimated symbol entropy. However, a repeating order provides a prediction rule that counting individual symbols does not capture.

These are different opportunities for compression. Symbol probabilities help us choose efficient codewords. Relationships between symbols can reduce the uncertainty that remains when the surrounding sequence is known. Both approaches can preserve the decoded sample values exactly.


### 6.2 Redundancy in Symbol Encoding

Return to the eight three-bit symbols and their unequal probabilities in Section 5.3 of the entropy notebook: `000` occurs with probability $1/2$, `001` with $1/4$, `010` with $1/8$, and each remaining symbol with $1/40$. Their symbol entropy is $H(X)\approx2.04024$ bit per source symbol.

For a code with average length $L$, we call the gap above the symbol-entropy bound the **absolute coding redundancy**. Since both $L$ and $H(X)$ are measured in bits per source symbol, their difference has the same unit:

$$
L-H(X)\quad\text{bit per source symbol}.
$$

For the original fixed-length code, this gives

$$
L-H(X)=3-2.04024\approx0.960\ \text{bit per source symbol}.
$$

The **relative coding redundancy** divides this gap by the average encoded length: $(L-H(X))/L=1-H(X)/L$. The units cancel, so relative redundancy is **dimensionless**, usually expressed as a percentage. For the fixed-length code,

$$
1-\frac{H(X)}{L}=1-\frac{2.04024}{3}\approx0.320=32.0\%.
$$

The probability-based code in the entropy notebook has $L=2.05$, leaving a much smaller gap of approximately $0.010$ bit per source symbol. Its actual saving relative to the three-bit code is about $31.7\%$, close to the bound above.

These comparisons use an independent-symbol model and exclude headers and the description of the code. A fractional average does not mean that an individual codeword contains a fraction of a bit. Practical codes use whole bits, and a short message need not attain the entropy bound. When symbol order is predictable, a model that uses that dependence can offer additional savings beyond those suggested by symbol frequencies alone.


### 6.3 Redundancy Across Symbols

#### 6.3.1 Sequential Redundancy

Sequential redundancy is predictable structure in the order of source symbols. Knowing earlier symbols, or a rule describing their arrangement, can reduce the uncertainty about the next symbol. This differs from redundancy due to unequal symbol probabilities: symbol frequencies alone do not describe which symbols follow which. The useful context may include one previous symbol, several previous symbols, or the position within a repeating pattern.

For example, `000 000 111 111 ...` repeats a four-symbol block. Knowing only that the current symbol is `000` does not tell us whether the next symbol is `000` or `111`; knowing the position within the block does. A lossless representation can exploit that rule instead of describing each occurrence separately.

Consider two sources that use only the three-bit symbols `000` and `111`:

- Source A chooses each symbol independently, with probability $1/2$ for either value.
- Source B alternates: `000 111 000 111 ...`. Assume either starting symbol is equally likely.

For both sources, one symbol considered alone has entropy $H(X_n)=1$ bit per source symbol. Both still use three bits per symbol in the original fixed-length representation. The difference between the sources appears when the previous symbol is known.

**Conditional entropy** $H(X_n\mid X_{n-1})$ measures the average uncertainty about the current symbol after the previous symbol is known. For Source A, that knowledge does not help, so the conditional entropy is one bit per source symbol. For Source B, it determines the next symbol exactly, so the conditional entropy is zero. In general,

$$
H(X_n\mid X_{n-1})\le H(X_n).
$$

The table compares the two sources. Both quantities are measured in **bits per source symbol**.

| Source | Symbol entropy $H(X_n)$ | Conditional entropy $H(X_n\mid X_{n-1})$ |
|:---|---:|---:|
| A: independent choices of `000` and `111`, each with probability $1/2$ | $1$ | $1$ |
| B: alternating `000 111 000 111 ...`, with either starting symbol equally likely | $1$ | $0$ |

Conditioning cannot increase uncertainty on average. The alternating source therefore has sequential redundancy that its individual-symbol probabilities do not reveal. Describing its rule, starting symbol, and length can avoid transmitting every symbol separately. The initial information still has to be represented; zero conditional entropy does not mean that a finite message requires no description.

#### 6.3.2 Mutual Information

A simple measure of the information about $X_n$ provided by the previous symbol is

$$I(X_n;X_{n-1})=H(X_n)-H(X_n\mid X_{n-1}).$$

This quantity is called **mutual information**. It uses the same letter $I$ as the information content $I(E)$ of a single event, introduced earlier. The notation $I(X;Y)$, with two variables separated by a semicolon, denotes mutual information rather than the information content of one event. With logarithms to base 2, mutual information is measured in **bits**. Here, we report it per adjacent symbol pair. Mutual information cannot be negative. In general, $0\leq I(X_n;X_{n-1})\leq\min\{H(X_n),H(X_{n-1})\}$. For the two-symbol sources used here, each entropy is at most 1 bit, so the mutual information lies between 0 and 1 bit per adjacent symbol pair. With a larger alphabet, its maximum can be greater than 1 bit.

In this context, mutual information measures the predictable information shared by two successive symbols. If the symbols are independent, it is zero. If the previous symbol completely determines the next one, it equals $H(X_n)$.

The difference $H(X_n)-H(X_n\mid X_{n-1})$ also quantifies how much sequential predictability is captured by **one previous symbol**, in bits per current source symbol. It does not necessarily capture all sequential redundancy: some patterns require a longer history. This differs from the coding gap $L-H(X)$ in Section 6.2, which compares an encoded length with the symbol-entropy bound.

Zero mutual information means that the two adjacent symbols are independent. It does not imply that the whole sequence has no structure. For example, in the repeating pattern `000 000 111 111 ...`, assume a uniformly random starting position within the four-symbol cycle. After either symbol, the next is equally likely to be `000` or `111`, so adjacent mutual information is zero. Two previous symbols, however, determine the next symbol exactly.


#### 6.3.3 Examples: Independent, Alternating, and Biased Dependent Sources

The independent and alternating sources both assign probability $1/2$ to `000` and `111`, so one symbol considered alone has entropy $H(X_n)=1$ bit per source symbol. We calculate mutual information by subtracting the uncertainty that remains when the previous symbol is known.

**Independent source:** knowing the previous symbol does not help predict the next. The conditional entropy remains one bit per source symbol, so

$$
I(X_n;X_{n-1})=1-1=0\ \text{bit per adjacent symbol pair}.
$$

**Alternating source:** knowing the previous symbol determines the next exactly. The conditional entropy is zero, so

$$
I(X_n;X_{n-1})=1-0=1\ \text{bit per adjacent symbol pair}.
$$

The two sources have the same symbol entropy but different mutual information. The previous symbol removes no uncertainty for the independent source and all uncertainty for the alternating source. These are already average quantities; no further averaging is needed.

The graph below compares the two sources above with a third example: a **biased, dependent source** that tends to repeat its previous symbol. Its next-symbol probabilities are:

| Previous symbol | Next symbol `000` | Next symbol `111` |
|:---:|:---:|:---:|
| `000` | $0.9$ | $0.1$ |
| `111` | $0.3$ | $0.7$ |

The symbol probabilities are $P(\texttt{000})=0.75$ and $P(\texttt{111})=0.25$. These probabilities are **stationary**: they stay the same from one position to the next under the transition rules in the table. For example, the probability of the next symbol being `000` is $0.75\cdot0.9+0.25\cdot0.3=0.75$. We use these probabilities for the first symbol as well. Its symbol entropy is therefore

$$
H(X_n)=-0.75\log_2(0.75)-0.25\log_2(0.25)\approx0.811\ \text{bit per source symbol}.
$$

Knowing the previous symbol reduces the uncertainty further. Weighting the uncertainty in each row of the table by the probability of that previous symbol gives

$$
H(X_n\mid X_{n-1})=
0.75\bigl[-0.9\log_2(0.9)-0.1\log_2(0.1)\bigr]
+0.25\bigl[-0.3\log_2(0.3)-0.7\log_2(0.7)\bigr]
\approx0.572\ \text{bit per source symbol}.
$$

For this third source, both bars lie between zero and one. The blue bar is below one because the symbols are not equally likely. The orange bar is lower still because the previous symbol helps predict the next, without determining it completely. Their difference is the mutual information, approximately $0.239$ bit per adjacent symbol pair. The graph uses estimates from simulated sequences, so its values can differ slightly from these theoretical results.

<p style="text-align: center;"><img src="../../.admin/img/mutual_information.svg" alt="Entropy and one-previous-symbol conditional entropy for independent, alternating, and biased dependent sources, with three-bit symbol sequences" style="max-width: 100%; height: auto;"><br><em>Example sequences: each three-bit word is one source symbol. The bars are estimated from 10,000 symbols per source; only the first 16 are shown.</em></p>


### 6.4 From Redundancy to Correlation and Source Coding

Unequal symbol probabilities and predictable symbol order can both offer opportunities for lossless compression. Symbol entropy describes uncertainty about one symbol considered alone. Conditional entropy describes the uncertainty that remains when another symbol is known. The average encoded length describes how many bits a particular code actually uses.

The next notebook, [Correlation](01.7_Correlation.ipynb), introduces tools for detecting certain relationships between numerical sample values. Such relationships may reveal redundancy, although correlation cannot detect every kind of dependence. The later [Source Coding](../02_Data_Communication/02.2_Source_Coding.ipynb) notebook studies practical ways to exploit redundancy while preserving the data. Channel coding then adds structured redundancy for a different purpose: reliable transmission.


### 6.5 Distinguishing Related Concepts in Language Models

The terms **tokenization**, **embeddings**, and **attention** also concern how data is represented and processed. They connect to the ideas in this notebook, but they describe different operations. The following distinctions help keep them separate.

**Tokenization is not an entropy calculation.** Tokenization divides text into source symbols called tokens, which may represent words, parts of words, or punctuation. In our sensor examples, we chose complete three-bit words as symbols; a tokenizer defines a different set of symbols for text. Once the symbols are defined, their probabilities can be used to calculate entropy. A token's integer ID is only a label, not its information content. Fewer tokens do not automatically mean lower entropy or a more efficient lossless representation.

**An embedding is not an information-content value or a compressed codeword.** An embedding maps a token ID to a learned numerical vector used by the model. Its coordinates help the model process relationships between tokens. They are not the quantity $I(x)=-\log_2p(x)$, and the vector's size does not state how many bits of information the token carries. Embeddings and the variable-length binary codewords in Section 6.2 serve different purposes.

**Attention is not conditional entropy or mutual information.** Attention is a computational mechanism that combines representations from different positions. In a model that predicts the next token, it can help use relevant preceding context. Attention weights are not measurements of the shared information $I(X;Y)$, nor are they the model's next-token probabilities. The model computes those probabilities from its resulting representations.

**The useful connection is predictability from context.** Consider the unfinished sentence:

> The cat sat on the ...

The preceding words help us judge which continuations are plausible, although they do not determine a unique answer. This is related to the sequential redundancy discussed above. If $T_n$ denotes the next token, then

$$
H(T_n\mid T_1,\ldots,T_{n-1})
$$

measures the average uncertainty remaining after the preceding tokens are known, in bits per token. Conditional entropy describes a property of the source distribution; attention is one mechanism a model can use to exploit context when estimating that distribution. The model's estimates can be imperfect.

Keep the roles distinct: **tokenization defines symbols, embeddings represent them, attention helps use context, and entropy quantifies uncertainty from probabilities.**


---

### 6.6 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**In the three-bit example, distinguish redundancy due to unequal symbol probabilities from redundancy due to a predictable symbol order. Which can be described using symbol frequencies alone, and which requires information about relationships between symbols?**

There are two different types of redundancy:

- Unequal symbol probabilities: Some symbols occur more frequently than others. This can be described using symbol frequencies alone. For example, if $000$ occurs with probability $1/2$, it can receive a shorter codeword than a symbol occurring with probability $1/40$.
- Predictable symbol order: The probability of a symbol depends on preceding symbols. This requires information about relationships between symbols, such as transition probabilities or repeating patterns.

For example, these two sequences have exactly the same frequencies—eight 000s and eight 111s:<br>
$Sequence\text{ }A: 000\text{ }111\text{ }000\text{ }111\text{ }000\text{ }111 ... ,\text{ }Sequence\text{ }B:\text{ }000\text{ }111\text{ }111\text{ }000\text{ }000\text{ }111 ...$<br>
Sequence A is much easier to predict because it alternates perfectly. Symbol counts alone cannot see that difference.

Symbol entropy $H(X_n)$ captures the first type but does not capture all sequential structure. Conditional entropy $H(X_n\mid X_{n-1})$ captures dependence on the immediately preceding symbol. More generally, conditional entropy based on longer histories can capture longer-range dependencies.

**For the alternating source `000 111 000 111 ...`, assume either starting symbol is equally likely. Calculate $H(X_n)$, $H(X_n\mid X_{n-1})$, and $I(X_n;X_{n-1})$. State the units and explain what each value means. Why is the symbol entropy not three bits, even though every symbol is represented by three bits?**

Each occurs with probability $1/2$:

$H(X_n)=-\left(\frac{1}{2}\log_2\frac{1}{2}+\frac{1}{2}\log_2\frac{1}{2}\right)=1$ bit/source symbol => before observing any context, there is uncertainty about whether the next symbol is 000 or 111.

Once the previous symbol is known, the next symbol is determined exactly because the source alternates between $000$ and $111$:

$H(X_n\mid X_{n-1})=0$ bit/source symbol => after seeing the previous symbol, there is no remaining uncertainty.

The mutual information is therefore:

$I(X_n;X_{n-1})=H(X_n)-H(X_n\mid X_{n-1})=1-0=1$ bit per adjacent symbol pair => the previous symbol provides one bit of useful information about the next symbol.

This means that before knowing the previous symbol, there is one bit of uncertainty about the next symbol. Once the previous symbol is known, there is no uncertainty because the next symbol is completely determined. Thus, the previous symbol provides one bit of information about the next symbol.

The symbol entropy is not three bits because entropy measures uncertainty, not the number of bits used to represent a symbol. Although each source symbol is physically represented by three bits, only two symbols ($000$ and $111$) can occur, and they are equally likely. Therefore, the uncertainty is only $\log_2 2=1$ bit/source symbol.

**A source repeats `000 000 111 111 ...`, with a uniformly random starting position within the four-symbol cycle. List the next symbol after each possible pair: `000 000`, `000 111`, `111 111`, and `111 000`. Does knowing one previous symbol determine the next? Do two previous symbols determine it? Use your answers to explain why this source has sequential redundancy.**

(answer here)
| Previous pair | Possible next symbol |
|---|---|
| `000 000` | `111` |
| `000 111` | `111` |
| `111 111` | `000` |
| `111 000` | `000` |

Knowing one previous symbol is insufficient:
- After 000, the next symbol can be either 000 or 111, depending on the position in the cycle.
- After 111, the next symbol can also be either 111 or 000.

But two previous symbols do determine the next symbol, see table above:

$H(X_n\mid X_{n-1})>0$, but $H(X_n\mid X_{n-2},X_{n-1})=0$

This source has sequential redundancy because the next symbol is not fully predictable from one previous symbol, but becomes perfectly predictable when two previous symbols are known. The individual symbol frequencies alone do not reveal this four-symbol cycle.

**What does $I(X_n;X_{n-1})=0$ tell us about two adjacent symbols? Does it prove that the entire sequence has no sequential redundancy? Explain using the independent source and the repeating four-symbol pattern as examples.**

Means that the previous symbol provides no information about the next symbol, on average. Equivalently,
$H(X_n\mid X_{n-1})=H(X_n)$.

For the independent source, the symbols 000 and 111 are chosen independently with probability 1/2. Therefore, <br>$H(X_n)=1,\quad H(X_n\mid X_{n-1})=1$,<br>
so<br>
$I(X_n;X_{n-1})=1-1=0$.

However, zero mutual information between adjacent symbols does not prove that the entire sequence has no structure.

For the repeating pattern
`000 000 111 111 ...`
one previous symbol does not determine the next, so its adjacent mutual information can also be zero. Nevertheless, two previous symbols determine the next symbol exactly: $H(X_n\mid X_{n-2},X_{n-1})=0$. Thus, the source contains sequential redundancy that is invisible when only adjacent pairs are considered. A longer history or a model of the repeating cycle is needed to detect it.

**For the biased, dependent source in Section 6.3.3, calculate the probability of the adjacent pair `111 111` using $P(X_{n-1}=\texttt{111})=0.25$ and the transition table. What would that pair's probability be if successive symbols were independent but still had $P(\texttt{111})=0.25$? Explain what the difference tells you about dependence. Then use the theoretical values $H(X_n)\approx0.811$ and $H(X_n\mid X_{n-1})\approx0.572$ to calculate the mutual information, and identify what it corresponds to in the bar graph.**

Hint: $P(a,b)=P(a)P(b\mid a)$. For independent symbols, $P(a,b)=P(a)P(b)$.

(answer here)

The source has $P(X_{n-1}=111)=0.25$ and the transition probability $P(X_n=111\mid X_{n-1}=111)=0.7$.<br>
Therefore, $P(X_{n-1}=111,X_n=111)=P(X_{n-1}=111)P(X_n=111\mid X_{n-1}=111)=0.25(0.7)=0.175$.<br>
So, $P(111,111)=0.175$.

If successive symbols were independent but still had $P(111)=0.25$, then $P(111,111)=P(111)P(111)=0.25^2=0.0625$.<br>
Thus, $P_{\text{independent}}(111,111)=0.0625$.<br>
The actual probability, 0.175, is much larger than 0.0625. This indicates positive dependence: after observing 111, the source is more likely to produce 111 again than it would be under independence.

Using the given theoretical entropy values, $H(X_n)=0.811$ and $H(X_n\mid X_{n-1})=0.572$.<br>
The mutual information is $I(X_n;X_{n-1})=H(X_n)-H(X_n\mid X_{n-1})=0.811-0.572=0.239$.<br>
Therefore, $I(X_n;X_{n-1})=0.239$ bit per adjacent symbol pair.

This is the vertical difference between the symbol-entropy bar and the conditional-entropy bar in the graph. It represents the average uncertainty about the current symbol that is removed by knowing the previous symbol.

**For the three-bit source in Section 6.2, use $H(X)=2.04024$ bit per source symbol. Calculate $L-H(X)$ and $1-H(X)/L$ for both the fixed-length code ($L=3$) and the probability-based code ($L=2.05$). Give the units or percentage for each result. Explain why the two codes have different average lengths even though they represent the same source.**

Given $H(X)=2.04024$.

Fixed-length code: 
- For $𝐿=3$ bits per symbol, $L-H(X)=3-2.04024=0.95976$. Thus, $L-H(X)=0.960$ bit per source symbol.
- The relative redundancy is $1-\frac{H(X)}{L}=1-\frac{2.04024}{3}\approx0.31992\approx32.0$%.

Probability-based code:
- For $L=2.05$ bits per symbol, $L-H(X)=2.05-2.04024=0.00976\approx0.010$ bit per source symbol.
- The relative redundancy is $1-\frac{H(X)}{L}=1-\frac{2.04024}{2.05}\approx0.00476\approx0.48$%.

| Code              | Average length `L` | Absolute redundancy `L-H(X)` | Relative redundancy |
| ----------------- | ------------------ | ---------------------------- | ------------------- |
| Fixed-length      | `3.00` bits/symbol | `0.960` bits/symbol          | `32.0\%`            |
| Probability-based | `2.05` bits/symbol | `0.010` bits/symbol          | `0.48\%`             |

Both codes represent the same source symbols, but they assign different codeword lengths. The fixed-length code uses three bits for every symbol, regardless of its probability. The probability-based code assigns short codewords to frequent symbols and long codewords to rare symbols, reducing the average length while preserving the source data exactly.

The average length is not required to be an integer because it is an expectation over many symbols. Individual codewords still contain whole numbers of bits, but their probability-weighted average can be fractional.

**OPTIONAL: Did you understand all tasks? Which explanation or exercise should be improved?**  

(please help the team by telling us what part was not understood, and we will work on it)
